## Phantom Positions — Cross-Model Comparison

Evaluates and compares three classification models (Logistic Regression, Random Forest, and SVM)
across three feature conditions (combined, text-only, and metadata-only) using the shared
preprocessing pipeline in `data/preprocessing.py`.

**Models:** Logistic Regression · Random Forest · LinearSVC  
**Feature Conditions:** combined · text_only · metadata_only  
**Metrics:** F1 (fake class), Precision, Recall, Accuracy, ROC AUC

All models use `random_state=5` and `max_features=5000` for TF-IDF to ensure
consistent, comparable results across conditions.

In [ ]:
import sys
sys.path.append('..')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import LinearSVC
from sklearn.metrics import (
    classification_report, confusion_matrix,
    roc_auc_score, roc_curve
)

from data.preprocessing import load_and_split

#### =============================================================
### 1. Model registry
####    SVM uses per-condition C values matching svm.ipynb
#### =============================================================

In [ ]:
CONDITIONS = ['combined', 'text_only', 'metadata_only']
MODEL_NAMES = ['Logistic Regression', 'Random Forest', 'SVM']

def get_model(model_name, condition):
    if model_name == 'Logistic Regression':
        return LogisticRegression(C=0.01, class_weight={0: 1, 1: 6}, random_state=5)
    elif model_name == 'Random Forest':
        return RandomForestClassifier(n_estimators=200, random_state=5)
    elif model_name == 'SVM':
        c_map = {'combined': 0.01, 'text_only': 0.001, 'metadata_only': 0.1}
        return LinearSVC(C=c_map[condition], random_state=5)

#### =============================================================
### 2. Run all 9 combinations and collect metrics
#### =============================================================

In [ ]:
results = []
roc_data = {}  # keyed by (model_name, condition) — used for ROC curves in section 8

for condition in CONDITIONS:
    print(f"\n--- Loading: {condition} ---")
    X_train, X_test, Y_train, Y_test = load_and_split(condition)

    for model_name in MODEL_NAMES:
        model = get_model(model_name, condition)
        model.fit(X_train, Y_train)
        Y_pred = model.predict(X_test)

        report = classification_report(Y_test, Y_pred, output_dict=True)

        if hasattr(model, 'predict_proba'):
            scores = model.predict_proba(X_test)[:, 1]
        else:
            scores = model.decision_function(X_test)

        auc = roc_auc_score(Y_test, scores)

        # Store scores separately — keeping arrays out of df_results prevents pivot errors
        roc_data[(model_name, condition)] = {'scores': scores, 'Y_test': Y_test}

        results.append({
            'Model':          model_name,
            'Condition':      condition,
            'Precision_fake': report['1']['precision'],
            'Recall_fake':    report['1']['recall'],
            'F1_fake':        report['1']['f1-score'],
            'Precision_real': report['0']['precision'],
            'Recall_real':    report['0']['recall'],
            'F1_real':        report['0']['f1-score'],
            'Accuracy':       report['accuracy'],
            'ROC_AUC':        auc,
        })

        print(f"  {model_name}: F1_fake={report['1']['f1-score']:.3f}, AUC={auc:.3f}")

df_results = pd.DataFrame(results)

#### =============================================================
### 3. Summary table — all 9 runs
#### =============================================================

In [ ]:
display_cols = ['Model', 'Condition', 'Precision_fake', 'Recall_fake',
                'F1_fake', 'Accuracy', 'ROC_AUC']

print("\n=== FULL COMPARISON TABLE ===")
print(df_results[display_cols].to_string(index=False))

#### =============================================================
### 4. Pivot table: F1 on fake class
#### =============================================================

In [ ]:
f1_pivot = df_results.pivot(index='Condition', columns='Model', values='F1_fake')
print("\n=== F1 Score (Fake Class) by Condition and Model ===")
print(f1_pivot.round(3))

#### =============================================================
### 5. Heatmap: F1 on fake class
#### =============================================================

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
sns.heatmap(f1_pivot, annot=True, fmt='.3f', cmap='YlOrRd',
            linewidths=0.5, ax=ax, vmin=0, vmax=1)
ax.set_title('F1 Score — Fake Class\nby Model and Feature Condition', fontsize=13)
ax.set_xlabel('')
ax.set_ylabel('')
plt.tight_layout()
plt.savefig('../outputs/heatmap_f1_fake.png', dpi=150)
plt.show()

#### =============================================================
### 6. Bar chart: F1 and Recall on fake class side-by-side
#### =============================================================

In [ ]:
x = np.arange(len(CONDITIONS))
width = 0.12
bar_colors = ['#4C72B0', '#DD8452', '#55A868']

fig, axes = plt.subplots(1, 2, figsize=(14, 5), sharey=False)

for i, (metric, ylabel, title) in enumerate([
    ('F1_fake',     'F1 Score', 'F1 Score — Fake Class'),
    ('Recall_fake', 'Recall',   'Recall — Fake Class'),
]):
    ax = axes[i]
    for j, (model_name, color) in enumerate(zip(MODEL_NAMES, bar_colors)):
        vals = [df_results[(df_results.Model == model_name) &
                           (df_results.Condition == c)]['F1_fake' if metric == 'F1_fake' else 'Recall_fake'].values[0]
                for c in CONDITIONS]
        ax.bar(x + j * width, vals, width, label=model_name, color=color)

    ax.set_xticks(x + width)
    ax.set_xticklabels(CONDITIONS, rotation=10)
    ax.set_ylabel(ylabel)
    ax.set_title(title)
    ax.set_ylim(0, 1.05)
    ax.legend()
    ax.grid(axis='y', alpha=0.3)

plt.suptitle('Model Comparison — Fake Posting Detection', fontsize=14, y=1.02)
plt.tight_layout()
plt.savefig('../outputs/bar_f1_recall_fake.png', dpi=150)
plt.show()

#### =============================================================
## 7. Confusion matrices — 3x3 grid (condition x model)
#### =============================================================

In [ ]:
fig, axes = plt.subplots(3, 3, figsize=(14, 12))

for row, condition in enumerate(CONDITIONS):
    X_train, X_test, Y_train, Y_test = load_and_split(condition)

    for col, model_name in enumerate(MODEL_NAMES):
        model = get_model(model_name, condition)
        model.fit(X_train, Y_train)
        Y_pred = model.predict(X_test)
        cm = confusion_matrix(Y_test, Y_pred)

        ax = axes[row][col]
        sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax,
                    xticklabels=['Real', 'Fake'], yticklabels=['Real', 'Fake'],
                    cbar=False)
        ax.set_title(f'{model_name}\n{condition}', fontsize=9)
        ax.set_xlabel('Predicted')
        ax.set_ylabel('Actual')

plt.suptitle('Confusion Matrices — All Models and Conditions', fontsize=14)
plt.tight_layout()
plt.savefig('../outputs/confusion_matrices_all.png', dpi=150)
plt.show()

#### =============================================================
### 8. Overlaid ROC curves — one plot per condition
#### =============================================================

In [ ]:
roc_colors = {'Logistic Regression': '#4C72B0', 'Random Forest': '#DD8452', 'SVM': '#55A868'}

fig, axes = plt.subplots(1, 3, figsize=(16, 5), sharey=True)

for ax, condition in zip(axes, CONDITIONS):
    for model_name in MODEL_NAMES:
        d = roc_data[(model_name, condition)]
        fpr, tpr, _ = roc_curve(d['Y_test'], d['scores'])
        auc = df_results[(df_results.Model == model_name) &
                         (df_results.Condition == condition)]['ROC_AUC'].values[0]
        ax.plot(fpr, tpr, label=f"{model_name} (AUC={auc:.3f})",
                color=roc_colors[model_name], linewidth=2)

    ax.plot([0, 1], [0, 1], 'k--', linewidth=0.8, alpha=0.5)
    ax.set_title(condition, fontsize=11)
    ax.set_xlabel('False Positive Rate')
    ax.set_ylabel('True Positive Rate')
    ax.legend(fontsize=8)
    ax.grid(alpha=0.3)

plt.suptitle('ROC Curves by Feature Condition', fontsize=14)
plt.tight_layout()
plt.savefig('../outputs/roc_curves_by_condition.png', dpi=150)
plt.show()